# Q1 — S&P 500 Stocks Added: Which Year Had Most Additions Since 2020?
**Homework:** `cohorts/2026/homework1.md:6-32` — *Which year had the highest number of S&P 500 additions since 2020?*

**Learning goals (novice):** Fetch a Wikipedia table with `requests` + `pandas.read_html`, parse dates, groupby year.

**Answer (verified 2026-09-14):** **2025 with 18 additions** (2024:16, 2022/23:15, 2026:13 partial). Additional: 224 stocks >20y exact cutoff (218 calendar strict).


## 1.1 What we will do

- Download Wikipedia S&P 500 table (hint code with headers)
- Keep `Symbol`, `Security`, `Date added` → extract `year_added`
- Count per year from 2020 → find max
- *Novice tip:* Wikipedia returns HTML; `StringIO(response.text)` avoids `lxml` file-path confusion.

In [ ]:
import requests
import pandas as pd
from io import StringIO

URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
response = requests.get(URL, headers=HEADERS)
response.raise_for_status()
print(f"HTTP {response.status_code}, {len(response.text)} chars")

# pandas reads all tables; first is S&P 500 list (503 rows)
tables = pd.read_html(StringIO(response.text))
print(f"Found {len(tables)} tables")
df = tables[0]
print(df.shape, list(df.columns))
df.head(3)

## 1.2 Clean dates and extract year

- `Date added` → datetime → `.dt.year`
- Check for missing dates (`NaT`) — should be 0.

In [ ]:
df["Date added"] = pd.to_datetime(df["Date added"], errors="coerce")
print("NaT count:", df["Date added"].isna().sum())
df["year_added"] = df["Date added"].dt.year
df[["Symbol","Security","Date added","year_added"]].head()

## 1.3 Count additions per year (from 2020)

- `value_counts().sort_index()` → counts per year
- Filter `>=2020` → `sort_values(ascending=False)` to find max.

In [ ]:
counts = df["year_added"].value_counts().sort_index()
print("All years tail:\n", counts.tail(10).to_string())
from_2020 = counts[counts.index >= 2020].sort_values(ascending=False)
print("\nFrom 2020 sorted by count:\n", from_2020.to_string())
print("\nFrom 2020 sorted by year:\n", counts[counts.index >= 2020].sort_index().to_string())

## 1.4 Answer: highest year since 2020

- `idxmax()` gives year with most additions.

In [ ]:
max_year = int(from_2020.idxmax())
max_count = int(from_2020.max())
print(f"Answer Q1: {max_year} with {max_count} stocks")
# Show which stocks
added_max = df[df["year_added"]==max_year][["Symbol","Security","Date added"]].sort_values("Date added")
print(added_max.to_string(index=False))
print(f"Includes hint examples: DASH, WSM, EXE, TKO on 2025-03-24?", "DASH" in added_max["Symbol"].values)

## 1.5 Additional: >20 years in index (as of 2026)

- Exact cutoff = Wikipedia last edit `2026-09-04` minus 20y → `2006-09-04`
- Calendar strict = `year <2006` (i.e. ≤2005)

In [ ]:
ref_date = pd.Timestamp("2026-09-04")
cutoff = ref_date - pd.DateOffset(years=20)
print(f"Cutoff {cutoff.date()}")
print("Exact < cutoff:", (df["Date added"] < cutoff).sum())
print("year <2006 (≤2005):", (df["year_added"] < 2006).sum())
print("year ≤2006:", (df["year_added"] <= 2006).sum())